In [0]:
# import this before each notebook

import re
import uuid
from datetime import date, datetime, timedelta

from delta.tables import DeltaTable
from datetime import date as dt_date

from pyspark.sql.functions import abs as spark_abs
from pyspark.sql.functions import max as spark_max
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.functions import min as spark_min

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_date, to_timestamp,
    regexp_extract, regexp_replace, split, trim, initcap,
    when, coalesce, round as spark_round,
    year, month, quarter, hour, dayofweek,
    from_unixtime, expr, length, md5, concat_ws,
    lower, input_file_name
)
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType,
    StringType, BooleanType
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType,
    StringType, BooleanType, DateType
)
from delta.tables import DeltaTable
from pyspark.sql.types import *


#### 1. S3 bucket path

In [0]:

# 1 S3 bucket
BUCKET = "retailflow360-capstone3"

# Raw
S3_RAW_BASE           = f"s3://{BUCKET}/raw"
S3_RAW_ELECTRONICS    = f"{S3_RAW_BASE}/electronics/"
S3_RAW_LIQUOR         = f"{S3_RAW_BASE}/liquor/Liquor_Sales.csv"
S3_RAW_BOOKS_DATA     = f"{S3_RAW_BASE}/books/books_data.csv"
S3_RAW_BOOKS_RATING   = f"{S3_RAW_BASE}/books/Books_rating.csv"

# Bronze
S3_BRONZE_BASE         = f"s3://{BUCKET}/bronze"
S3_BRONZE_ELECTRONICS  = f"{S3_BRONZE_BASE}/electronics/"
S3_BRONZE_LIQUOR       = f"{S3_BRONZE_BASE}/liquor_sales/"
S3_BRONZE_BOOKS_DATA   = f"{S3_BRONZE_BASE}/books_data/"
S3_BRONZE_BOOKS_RATING = f"{S3_BRONZE_BASE}/books_rating/"

# Silver
S3_SILVER_BASE         = f"s3://{BUCKET}/silver"
S3_SILVER_ELECTRONICS  = f"{S3_SILVER_BASE}/electronics"
S3_SILVER_LIQUOR       = f"{S3_SILVER_BASE}/liquor_sales"
S3_SILVER_BOOKS_DATA   = f"{S3_SILVER_BASE}/books_data"
S3_SILVER_BOOKS_RATING = f"{S3_SILVER_BASE}/books_rating"

# DQ
S3_DQ_BASE       = f"s3://{BUCKET}/dq"
S3_DQ_LOGS       = f"{S3_DQ_BASE}/dq_logs"
S3_Q_ELEC        = f"{S3_DQ_BASE}/electronics_quarantine"
S3_Q_LIQ         = f"{S3_DQ_BASE}/liquor_sales_quarantine"
S3_Q_BD          = f"{S3_DQ_BASE}/books_data_quarantine"
S3_Q_BR          = f"{S3_DQ_BASE}/books_rating_quarantine"

# Gold
S3_GOLD_BASE           = f"s3://{BUCKET}/gold"
S3_GOLD_DIM_DATE       = f"{S3_GOLD_BASE}/dim_date"
S3_GOLD_DIM_DOMAIN     = f"{S3_GOLD_BASE}/dim_domain"
S3_GOLD_DIM_PRODUCT    = f"{S3_GOLD_BASE}/dim_product"
S3_GOLD_DIM_GEOGRAPHY  = f"{S3_GOLD_BASE}/dim_geography"
S3_GOLD_DIM_STORE      = f"{S3_GOLD_BASE}/dim_store"
S3_GOLD_DIM_CUSTOMER   = f"{S3_GOLD_BASE}/dim_customer"
S3_GOLD_FACT_SALES     = f"{S3_GOLD_BASE}/fact_sales"

# platinm
S3_PLATINUM_BASE              = f"s3://{BUCKET}/platinum"
S3_PLT_EXECUTIVE_SUMMARY      = f"{S3_PLATINUM_BASE}/agg_executive_summary"
S3_PLT_TIME_SERIES            = f"{S3_PLATINUM_BASE}/agg_time_series"
S3_PLT_PRODUCT_INSIGHTS       = f"{S3_PLATINUM_BASE}/agg_product_insights"
S3_PLT_GEOGRAPHY              = f"{S3_PLATINUM_BASE}/agg_geography"
S3_PLT_CUSTOMER_INSIGHTS      = f"{S3_PLATINUM_BASE}/agg_customer_insights"
S3_PLT_REVIEWS_RATINGS        = f"{S3_PLATINUM_BASE}/agg_reviews_ratings"


#### 2. unity catalog path

In [0]:

# 2 Unity Catalog 
CATALOG        = "retailflow360"
BRONZE_SCHEMA  = "bronze"
SILVER_SCHEMA  = "silver"
DQ_SCHEMA      = "dq"
GOLD_SCHEMA    = "gold"
PLATINUM_SCHEMA = "platinum"

# Bronze tables
TBL_BRZ_ELECTRONICS  = f"{CATALOG}.{BRONZE_SCHEMA}.electronics"
TBL_BRZ_LIQUOR       = f"{CATALOG}.{BRONZE_SCHEMA}.liquor_sales"
TBL_BRZ_BOOKS_DATA   = f"{CATALOG}.{BRONZE_SCHEMA}.books_data"
TBL_BRZ_BOOKS_RATING = f"{CATALOG}.{BRONZE_SCHEMA}.books_rating"

# Silver tables
TBL_SLV_ELECTRONICS  = f"{CATALOG}.{SILVER_SCHEMA}.electronics"
TBL_SLV_LIQUOR       = f"{CATALOG}.{SILVER_SCHEMA}.liquor_sales"
TBL_SLV_BOOKS_DATA   = f"{CATALOG}.{SILVER_SCHEMA}.books_data"
TBL_SLV_BOOKS_RATING = f"{CATALOG}.{SILVER_SCHEMA}.books_rating"

# DQ tables
TBL_DQ_LOGS  = f"{CATALOG}.{DQ_SCHEMA}.dq_logs"
TBL_Q_ELEC   = f"{CATALOG}.{DQ_SCHEMA}.electronics_quarantine"
TBL_Q_LIQ    = f"{CATALOG}.{DQ_SCHEMA}.liquor_sales_quarantine"
TBL_Q_BD     = f"{CATALOG}.{DQ_SCHEMA}.books_data_quarantine"
TBL_Q_BR     = f"{CATALOG}.{DQ_SCHEMA}.books_rating_quarantine"

# Gold tables
TBL_DIM_DATE      = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"
TBL_DIM_DOMAIN    = f"{CATALOG}.{GOLD_SCHEMA}.dim_domain"
TBL_DIM_PRODUCT   = f"{CATALOG}.{GOLD_SCHEMA}.dim_product"
TBL_DIM_GEOGRAPHY = f"{CATALOG}.{GOLD_SCHEMA}.dim_geography"
TBL_DIM_STORE     = f"{CATALOG}.{GOLD_SCHEMA}.dim_store"
TBL_DIM_CUSTOMER  = f"{CATALOG}.{GOLD_SCHEMA}.dim_customer"
TBL_FACT_SALES    = f"{CATALOG}.{GOLD_SCHEMA}.fact_sales"

# platinum tables
TBL_PLT_EXECUTIVE_SUMMARY     = f"{CATALOG}.{PLATINUM_SCHEMA}.agg_executive_summary"
TBL_PLT_TIME_SERIES           = f"{CATALOG}.{PLATINUM_SCHEMA}.agg_time_series"
TBL_PLT_PRODUCT_INSIGHTS      = f"{CATALOG}.{PLATINUM_SCHEMA}.agg_product_insights"
TBL_PLT_GEOGRAPHY             = f"{CATALOG}.{PLATINUM_SCHEMA}.agg_geography"
TBL_PLT_CUSTOMER_INSIGHTS     = f"{CATALOG}.{PLATINUM_SCHEMA}.agg_customer_insights"
TBL_PLT_REVIEWS_RATINGS       = f"{CATALOG}.{PLATINUM_SCHEMA}.agg_reviews_ratings"


#### 3. run identity

In [0]:

# Run identity (generated fresh every execution)
BATCH_ID       = str(uuid.uuid4())
INGESTION_DATE = str(date.today())
RUN_ID         = str(uuid.uuid4())
RUN_TS         = datetime.utcnow()
RUN_DT         = str(date.today())
RUN_DATE       = str(date.today())

# SCD2 sentinel 
SCD2_MAX_DATE = "9999-12-31"

# Spark config
spark.conf.set("spark.sql.files.maxPartitionBytes", "134217728")
#spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "60")


/home/spark-b8b8cdce-dc3f-4553-a7e9-8c/.ipykernel/1866/command-8628948029972694-2015145426:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TS         = datetime.utcnow()
